# 🚀 Master Notebook: Huấn Luyện AI 2.0 Multi-Agent RAG Trên Kaggle Dual T4
### Tích hợp công thức SOTA từ Unsloth (Qwen-Embedding-4B), MoAI, CoREB và Turkish Legal RAG
- **Phần cứng**: Kaggle Dual Tesla T4 (`2 x 16GB = 32GB VRAM`)
- **Tối ưu**: `CachedMultipleNegativesRankingLoss` (GradCache) + LoRA FP16
- **Tiêu chuẩn**: Đánh giá thực nghiệm độc lập bằng `Recall@10`, `nDCG@10`, `MRR@10`


In [ ]:
def patch_environment():
    """Patches missing triton.ops, language attributes, and handles CPU/GPU fallbacks for Kaggle and Colab."""
    import os
    import sys
    import types
    import importlib.machinery

    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 1. Patch Triton mock safely
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__spec__ = importlib.machinery.ModuleSpec("triton", None)
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    # Ensure triton.language exists to satisfy transformers.modeling_utils import checks
    if not hasattr(m_triton, "language"):
        m_lang = types.ModuleType("triton.language")
        m_lang.__spec__ = importlib.machinery.ModuleSpec("triton.language", None)
        m_lang.constexpr = lambda x: x
        setattr(m_triton, "language", m_lang)
        sys.modules["triton.language"] = m_lang

    if not hasattr(m_triton, "cdiv"):
        setattr(m_triton, "cdiv", lambda a, b: (a + b - 1) // b)
    if not hasattr(m_triton, "jit"):
        setattr(m_triton, "jit", lambda *a, **k: (lambda f: f))

    m_ops = types.ModuleType("triton.ops")
    m_ops.__spec__ = importlib.machinery.ModuleSpec("triton.ops", None)
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.__spec__ = importlib.machinery.ModuleSpec("triton.ops.matmul_perf_model", None)
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 2. Bulletproof fallback for bitsandbytes
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes", None)
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")
        m_bnb_nn.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn", None)

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb_nn.modules.__spec__ = importlib.machinery.ModuleSpec("bitsandbytes.nn.modules", None)
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.47.0"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')
else:
    print('⚠️ CẢNH BÁO: BẠN CHƯA BẬT GPU!')
    print('👉 Vui lòng vào menu Runtime -> Change runtime type -> Chọn T4 GPU (hoặc Dual T4 trên Kaggle) -> Nhấn Save và chạy lại từ đầu.')


In [ ]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate>=1.1.0 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops huggingface_hub

print('✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!')


In [ ]:
# BƯỚC 3: HUẤN LUYỆN CROSS-ENCODER RERANKER (RECIPE CoREB: LoRA r=16, alpha=16, nDCG@10)
import os, sys

# 1. Đảm bảo mã nguồn training_lab có sẵn
if not os.path.exists("/kaggle/working/training_lab"):
    print("⏳ Đang đồng bộ mã nguồn từ GitHub...", flush=True)
    !git clone https://github.com/nguyentatmanh/Research-assist.git /kaggle/working/Research-assist
    !cd /kaggle/working/Research-assist && (git checkout upgrade 2>/dev/null || true)
    !cp -r /kaggle/working/Research-assist/training_lab /kaggle/working/
    print("✅ Đã sẵn sàng training_lab!")

# 2. Chạy huấn luyện Reranker Cross-Encoder
py_bin = sys.executable
print('\n' + '='*80 + '\n🎯 [STAGE 1] HUẤN LUYỆN RERANKER CROSS-ENCODER VỚI HARD NEGATIVES\n' + '='*80)
mod = "training_lab.training.reranker.train_reranker_lora" if os.path.exists("training_lab") else "training.reranker.train_reranker_lora"
dpath = "training_lab/data/triplets/massive_reranker_pairs.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_reranker_pairs.jsonl"

!{py_bin} -m {mod} --epochs 3 --lr 3e-5 --data {dpath}


In [ ]:
import os, shutil
from pathlib import Path
from IPython.display import FileLink

os.chdir('/kaggle/working')
reranker_dir = Path("/kaggle/working/training_lab/training/reranker/weights")
zip_name = "Reranker_LoRA_Best_DualT4.zip"
if reranker_dir.exists():
    shutil.make_archive(zip_name.replace(".zip", ""), "zip", reranker_dir)
    print(f"🎉 Đã đóng gói thành công: {zip_name}")
    display(FileLink(zip_name))
else:
    print("⚠️ Chưa tìm thấy weights Reranker.")


In [ ]:
# ==============================================================================
# BƯỚC 4: HUẤN LUYỆN EMBEDDING (RECIPE MoAI: 1 EPOCH SOTA)
# ==============================================================================
import os, sys

dpath = None
for cand in [
    "/kaggle/working/training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "training_lab/data/triplets/massive_embedding_triplets.jsonl"
]:
    if os.path.exists(cand):
        dpath = cand
        break

if not dpath:
    print("⏳ Chưa có dữ liệu, đang clone repository từ GitHub...", flush=True)
    !rm -rf /kaggle/working/temp_repo
    !git clone https://github.com/nguyentatmanh/Research-assist.git /kaggle/working/temp_repo
    !cd /kaggle/working/temp_repo && (git checkout upgrade 2>/dev/null || true)
    !cp -rn /kaggle/working/temp_repo/training_lab /kaggle/working/
    !rm -rf /kaggle/working/temp_repo
    dpath = "/kaggle/working/training_lab/data/triplets/massive_embedding_triplets.jsonl"
    print("✅ Đã nạp thành công bộ dữ liệu massive_embedding_triplets.jsonl!")

py_bin = sys.executable
print('\n' + '='*80 + '\n⚡ [STAGE 2] HUẤN LUYỆN EMBEDDING 1 EPOCH CHUẨN VÀNG SOTA (~2.5 TIẾNG)\n' + '='*80)
os.chdir("/kaggle/working")
mod = "training_lab.training.embedding.train_embedding_lora"
!{py_bin} -m {mod} --epochs 1 --lr 3e-5 --data {dpath} --batch-size 8


In [ ]:
import os, shutil
from pathlib import Path
from IPython.display import FileLink

os.chdir('/kaggle/working')
embed_dir = None
for cand in [
    Path('/kaggle/working/training_lab/training/embedding/weights'),
    Path('/kaggle/working/training_lab/models/adapters/embedding_lora'),
    Path('training_lab/training/embedding/weights')
]:
    if cand.exists():
        embed_dir = cand
        break

zip_name = "Embedding_LoRA_Best_DualT4.zip"
if embed_dir:
    shutil.make_archive(zip_name.replace(".zip", ""), "zip", embed_dir)
    print(f"🎉 Đã đóng gói thành công: {zip_name}")
    display(FileLink(zip_name))
else:
    print("⚠️ Chưa tìm thấy weights Embedding.")


In [ ]:
import os, sys, shutil
from pathlib import Path
from IPython.display import FileLink

print("=" * 80)
print("📝 HUẤN LUYỆN AGENT 5 REPORT WRITER TRÊN KAGGLE DUAL T4")
print("=" * 80)

target_script = "/kaggle/working/training_lab/agent5_writer/train_agent5_t4x2.py"
if os.path.exists(target_script):
    !python {target_script} --epochs-sft 3 --epochs-dpo 2

    # Dọn dẹp checkpoints phụ để file zip tinh gọn
    !rm -rf /kaggle/working/training_lab/agent5_writer/weights/*checkpoints*
    !rm -rf /kaggle/working/training_lab/agent5_writer/weights/**/checkpoints*

    os.chdir('/kaggle/working')
    zip_out = "Agent5_Academic_Writer_DualT4.zip"
    weights_dir = Path("/kaggle/working/training_lab/agent5_writer/weights")
    if weights_dir.exists():
        shutil.make_archive(zip_out.replace(".zip", ""), "zip", weights_dir)
        print(f"\n🎉 Đã hoàn thành huấn luyện và đóng gói: {zip_out}")
        display(FileLink(zip_out))
else:
    print(f"ℹ️ File {target_script} không tồn tại.")
